# PAPI - Tìm hiểu dữ liệu thô

Mục tiêu của notebook này là mô tả nguồn gốc, cấu trúc và các đặc điểm cần lưu ý của bộ dữ liệu PAPI trước khi tiến hành tiền xử lý.

Thứ tự thực hiện: notebook này được chạy trước, sau đó đến `preprocessing.ipynb` và `eda.ipynb`.

Bộ dữ liệu PAPI (Chỉ số Hiệu quả Quản trị và Hành chính công cấp tỉnh) do UNDP phối hợp với CECODES và RTA thực hiện, đo lường cảm nhận của người dân về chính quyền địa phương tại 63 tỉnh, thành trong giai đoạn 2011-2024. Tài liệu chuẩn mô tả đầy đủ: `docs/data/README.md`.

## 1. Danh mục tệp dữ liệu thô

In [1]:
import sys, os
ROOT = os.path.dirname(os.getcwd())  # tu notebooks/ len thu muc goc
sys.path.insert(0, os.path.join(ROOT, 'src'))
import pandas as pd
pd.set_option('display.max_columns', 30, 'display.width', 160)
RAW = os.path.join(ROOT, 'data', 'raw'); PROC = os.path.join(ROOT, 'data', 'processed')
import glob
files = sorted(glob.glob(os.path.join(RAW, '*.xlsx')))
pd.DataFrame({'file': [os.path.basename(f) for f in files],
              'KB': [round(os.path.getsize(f) / 1024) for f in files]})

,file,KB
0,1.2021PAPI_ProvincialIndicators_BangChiTieuCap...,472
1,2019_PAPI_Provincial_indicators2019_VIE_ENG.xlsx,542
2,2020PAPI_ProvincialIndicators_BangChiTieuCapTi...,346
3,2022PAPI_ProvincialIndicators_BangChiTieuCapTi...,858
4,2023PAPI_ProvincialIndicators_BangChiTieuCapTi...,1525
5,2024PAPI_ProvincialIndicators_BangChiTieuCapTi...,1905
6,PAPI-2011-Dữ-liệu-1.xlsx,52
7,PAPI-2012-Dữ-liệu-1.xlsx,52
8,PAPI-2013-Dữ-liệu-1.xlsx,52
9,PAPI-2014-Dữ-liệu-1.xlsx,52


**Nhận xét.** Bộ dữ liệu gồm 14 tệp Excel, được đặt tên theo ba thời kỳ tương ứng với ba cách tổ chức khác nhau. Đơn vị quan sát là cấp tỉnh đã tổng hợp, không phải dữ liệu khảo sát từng cá nhân.

## 2. Cấu trúc điểm theo bốn cấp

Điểm PAPI được tổ chức theo bốn cấp, từ khái quát đến chi tiết: tổng điểm PAPI (thang 10-80), tám trục nội dung (thang 1-10), các trục thành phần, và các chỉ tiêu gốc. Điểm mỗi trục bằng tổng các trục thành phần; tổng PAPI bằng tổng tám trục.

## 3. Thời kỳ 2011-2017: tỉnh theo hàng

In [2]:
f16 = os.path.join(RAW, 'PAPI-2016-Dữ-liệu-1.xlsx')
sh = [s for s in pd.ExcelFile(f16).sheet_names if 'tổng hợp' in s][0]
d16 = pd.read_excel(f16, sheet_name=sh)
d16.columns = [str(c).strip() for c in d16.columns]
cols = [d16.columns[0]] + [c for c in d16.columns if c[:2] in ('1:','2:','3:','4:','5:','6:')]
d16[cols].head(4)

,Tên tỉnh,1: Tham gia của người dân ở cấp cơ sở,"2: Công khai, minh bạch trong việc ra quyết định",3: Trách nhiệm giải trình với người dân,4: Kiểm soát tham nhũng trong khu vực công,5: Thủ tục hành chính công,6: Cung ứng dịch vụ công
0,Hà Nội,5.337698,5.076181,4.261384,5.238945,7.087234,6.804520
1,Hà Giang,5.339402,5.274486,4.403687,5.823136,6.635267,6.481400
2,Cao Bằng,5.207546,5.499399,4.442169,5.529461,7.018959,6.626322
3,Bắc Kạn,5.351490,5.446347,5.051856,5.896996,7.186898,6.678397


**Nhận xét.** Trong các tệp giai đoạn 2011-2017, mỗi dòng tương ứng với một tỉnh và mỗi cột là một trục hoặc chỉ số. Giai đoạn này chỉ có sáu trục nội dung; hai trục Quản trị môi trường và Quản trị điện tử chưa được đưa vào.

## 4. Thời kỳ 2018-2024: tỉnh theo cột

In [3]:
f23 = os.path.join(RAW, '2023PAPI_ProvincialIndicators_BangChiTieuCapTinh..xlsx')
d23 = pd.read_excel(f23, sheet_name='2023_VIE_ENG', header=None)
d23.iloc[:11, [0, 1, 2, 3, 4, 5]]

,0,1,2,3,4,5
0,Mã tỉnh,Mã tỉnh,Thang điểm/Scale,1,2,4
1,Tỉnh/Thành phố trực thuộc trung ương,Tỉnh/Thành phố trực thuộc trung ương,NaN,Hà Nội,Hà Giang,Cao Bằng
2,Unweighted PAPI Score,Chỉ số PAPI tổng hợp (không có trọng số),10-80 điểm/points,43.960304,44.247929,41.655148
3,Unweighted 95% CI Low,Chỉ số PAPI tổng hợp (không có trọng số) - điể...,NaN,43.700916,42.789047,41.337597
4,Unweighted 95% CI High,Chỉ số PAPI tổng hợp (không có trọng số) - điể...,NaN,44.219692,45.70681,41.972698
5,Unweighted Standard Error,Chỉ số PAPI tổng hợp (không có trọng số) - sai...,NaN,0.157685,0.886877,0.193045
6,Dimension 1: Participation,Chỉ số nội dung 1: Tham gia của người dân ở cấ...,1-10 điểm/points,5.427534,5.283103,4.772978
7,Civic Knowledge,1.1: Tri thức công dân,0.25-2.5 điểm,1.323925,1.021637,0.937442
8,Proportion with Civic Knowledge,Hiểu biết về chính sách hiện hành (%),0%-100%,0.745272,0.703545,0.483298
9,Knowledge of Leaders,Hiểu biết về vị trí lãnh đạo (%),0%-100%,0.581964,0.334127,0.369411


**Nhận xét.** Từ năm 2018, bảng được trình bày theo chiều ngược lại: mỗi dòng là một chỉ tiêu, mỗi cột là một tỉnh. Giai đoạn này có đủ tám trục và một dòng tổng (Unweighted PAPI Score). Tên tỉnh nằm ở một dòng tiêu đề; một số tệp có thêm dòng mã tỉnh.

## 5. Các trục thành phần

In [4]:
d1 = pd.read_excel(f16, sheet_name='Điểm thành phần 1')
print('Các cột của trục D1:')
for c in d1.columns:
    print(' -', c)

Các cột của trục D1:
 - Tên tỉnh 
 -   1: Tham gia của người dân ở cấp cơ sở 
 - 1.1: Tri thức công dân 
 - 1.2: Cơ hội tham gia 
 - 1.3: Chất lượng bầu cử
 - 1.4: Đóng góp tự nguyện 


**Nhận xét.** Mỗi sheet Điểm thành phần phân rã một trục thành các trục thành phần. Ví dụ trục Tham gia gồm Tri thức công dân, Cơ hội tham gia, Chất lượng bầu cử và Đóng góp tự nguyện. Phiên bản dữ liệu đầu tiên chỉ sử dụng đến cấp tám trục.

## 6. Các điểm cần lưu ý khi tiền xử lý

Quá trình khảo sát đã phát hiện tám điểm cần xử lý: (1) chiều trình bày đảo ngược giữa hai thời kỳ; (2) nhãn trục thay đổi từ tiếng Việt sang tiếng Anh kể từ năm 2020, do đó cần nhận diện trục theo số thứ tự 1-8 thay vì theo nhãn văn bản; (3) số trục tăng từ sáu lên tám kể từ năm 2018; (4) các tệp mới chứa lại dữ liệu năm cũ, cần chọn một nguồn chuẩn cho mỗi năm; (5) tệp năm 2024 được tổ chức lại theo 34 tỉnh, làm rỗng dữ liệu một số tỉnh ở các sheet năm cũ; (6) các năm gần đây có cả giá trị Weighted, khoảng tin cậy và sai số, trong đó chỉ sử dụng giá trị Unweighted; (7) một số tên tab ghi lệch năm; (8) tên tỉnh không nhất quán về dấu và tiền tố. Cách xử lý các điểm này được trình bày trong notebook preprocessing.ipynb.